In [1]:
from pathlib import Path
import torch 

dir2correct = Path("/data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_main_exp")
save_dir = dir2correct.parent / (dir2correct.stem + "_corrected")

print(f"Correcting paths in {dir2correct} and saving to {save_dir}")
save_dir.mkdir(parents=True, exist_ok=True)




Correcting paths in /data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_main_exp and saving to /data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_main_exp_corrected


In [2]:
import tqdm 

for split in ["train", "val", "test"]:
    split_dir = dir2correct / split
    stft_dir = dir2correct / "stft" / "STFT_fl0.064_fs0.016_sf16000_wtsqrt-hann" / split
    feat_dir = dir2correct / "features" / "WGMSC_nb_fl0.064_fs0.016_sf16000_winsqrt-hann_stc0.5_wtc0.5_stcr0.5_wtcr0.5" / split
    # sort all files in these directories
    raw_files = sorted(list(split_dir.rglob("*.pt")))
    
    for file_path in tqdm.tqdm(raw_files, desc=f"Correcting {split} split"):
        scenario = torch.load(file_path, weights_only=False)
        assert isinstance(scenario, dict), f"Scenario {file_path} is not a dict."
        # if key "raw_audio" is in scenario, check whether the key meta is a dict with a key "references". If yes remove the key "raw_audio"
        if scenario["input_type"] == "raw_audio" and "raw_audio" in scenario and "meta" in scenario:
            if isinstance(scenario["meta"], dict) and "references" in scenario["meta"]:
                del scenario["raw_audio"]
            else:
                raise ValueError(f"Scenario {file_path} has key 'raw_audio' but its 'meta' is not a dict with key 'references'.")
            new_meta = scenario["meta"].copy()
            if "gt_rtf_stream" in new_meta:
                del new_meta["gt_rtf_stream"]
            if "gt_ids_stream" in new_meta:
                del new_meta["gt_ids_stream"]
            if "id_map" in new_meta:
                del new_meta["id_map"]
            scenario["meta"] = new_meta
            save_path = save_dir / file_path.relative_to(dir2correct)
            save_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(scenario, save_path)
            # print(f"C: {file_path} - removing raw audio and unnecessary meta entries.")
            # print(f"S: {save_path}")
        else:
            raise ValueError(f"Scenario {file_path} has unexpected input_type {scenario['input_type']}.")
        for derived in ["stft", "features"]:
            derived_path = (
                dir2correct
                / derived
                / (
                    "STFT_fl0.064_fs0.016_sf16000_wtsqrt-hann" if derived == "stft" else "WGMSC_nb_fl0.064_fs0.016_sf16000_winsqrt-hann_stc0.5_wtc0.5_stcr0.5_wtcr0.5"
                )
                / split
                / file_path.name
            )
            derived_scenario = torch.load(derived_path, weights_only=False)
            assert isinstance(derived_scenario, dict), f"Derived scenario {derived_path} is not a dict."
            if derived_scenario["input_type"] == derived and "meta" in scenario:
                del derived_scenario["meta"]
                derived_save_path = save_dir / derived_path.relative_to(dir2correct)
                derived_save_path.parent.mkdir(parents=True, exist_ok=True)
                torch.save(derived_scenario, derived_save_path)
                # print(f"C: {derived_path} - removing meta.")
                # print(f"S: {derived_save_path}")
            else:
                raise ValueError(f"Scenario {file_path} has unexpected input_type {scenario['input_type']}.")

Correcting train split:   0%|          | 0/5280 [00:00<?, ?it/s]

Correcting test split: 100%|██████████| 2640/2640 [17:56<00:00,  2.45it/s]


In [3]:
from utilities import print_structure

# test newly created files
for split in ["train", "val", "test"]:
    # take a random raw file from the corrected directory
    split_dir = save_dir / split
    raw_files = sorted(list(split_dir.rglob("*.pt")))
    if not raw_files:
        continue
    test_file = raw_files[0]
    scenario = torch.load(test_file, weights_only=False)
    print(f"Testing corrected file {test_file}:")
    # print keys and keys of meta if exists
    if "meta" in scenario and isinstance(scenario["meta"], dict):
        print("Keys in scenario:", scenario.keys())
        print("Keys in meta:", scenario["meta"].keys())
    else:
        print("Keys in scenario:", scenario.keys())
    # print_structure(scenario)
    for derived in ["stft", "features"]:
        derived_path = (
            save_dir
            / derived
            / (
                "STFT_fl0.064_fs0.016_sf16000_wtsqrt-hann" if derived == "stft" else "WGMSC_nb_fl0.064_fs0.016_sf16000_winsqrt-hann_stc0.5_wtc0.5_stcr0.5_wtcr0.5"
            )
            / split
            / test_file.name
        )
        derived_scenario = torch.load(derived_path, weights_only=False)
        print(f"Testing corrected derived file {derived_path}:")
        if "meta" in derived_scenario and isinstance(derived_scenario["meta"], dict):
            print("Keys in derived scenario:", derived_scenario.keys())
            print("Keys in meta:", derived_scenario["meta"].keys())
        else:
            print("Keys in derived scenario:", derived_scenario.keys())
        # print_structure(derived_scenario)

Testing corrected file /data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_main_exp_corrected/train/scenario_0.pt:
Keys in scenario: dict_keys(['meta', 'input_type'])
Keys in meta: dict_keys(['scenario_params', 'source_count', 'scenario_id', 'references', 'rtfs', 'sad_samples', 'sad_frames', 'segments'])
Testing corrected derived file /data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_main_exp_corrected/stft/STFT_fl0.064_fs0.016_sf16000_wtsqrt-hann/train/scenario_0.pt:
Keys in derived scenario: dict_keys(['stft', 'input_type', 'stft_info'])
Testing corrected derived file /data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_main_exp_corrected/features/WGMSC_nb_fl0.064_fs0.016_sf16000_winsqrt-hann_stc0.5_wtc0.5_stcr0.5_wtcr0.5/train/scenario_0.pt:
Keys in derived scenario: dict_keys(['features', 'input_type', 'feature_info'])
Testing corrected file /data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_main_exp_corr